In [1]:
# Imports and path setup
from __future__ import annotations
import os
from pathlib import Path
from typing import Callable

import cv2
import numpy as np
from tqdm import tqdm

# Absolute project root
ROOT_DIR = Path('/Users/Kota/blended/Team3AmazonProject')

# IO directories
INPUT_DIR = ROOT_DIR / 'data' / 'mixed'
OUTPUT_DIR_GLOBAL = ROOT_DIR / 'data' / 'mixed_grobal'  # per request: "grobal"
OUTPUT_DIR_CLAHE = ROOT_DIR / 'data' / 'mixed_clahe'

# Supported image extensions
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

INPUT_DIR, OUTPUT_DIR_GLOBAL, OUTPUT_DIR_CLAHE


(PosixPath('/Users/Kota/blended/Team3AmazonProject/data/mixed'),
 PosixPath('/Users/Kota/blended/Team3AmazonProject/data/mixed_grobal'),
 PosixPath('/Users/Kota/blended/Team3AmazonProject/data/mixed_clahe'))

In [2]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def equalize_global_bgr(image_bgr: np.ndarray) -> np.ndarray:
    """Apply global histogram equalization on luminance channel (LAB.L) only."""
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    L_eq = cv2.equalizeHist(L)
    lab_eq = cv2.merge([L_eq, A, B])
    out_bgr = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)
    return out_bgr


def equalize_clahe_bgr(image_bgr: np.ndarray, clip_limit: float = 3.0, tile_grid_size: tuple[int, int] = (8, 8)) -> np.ndarray:
    """Apply CLAHE on luminance channel (LAB.L) only."""
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    L_eq = clahe.apply(L)
    lab_eq = cv2.merge([L_eq, A, B])
    out_bgr = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)
    return out_bgr


In [3]:
def process_dataset(
    input_dir: Path,
    output_dir: Path,
    transform: Callable[[np.ndarray], np.ndarray],
    overwrite: bool = False,
) -> None:
    """
    Walk input_dir recursively, transform images, and save to output_dir preserving structure.
    Non-image files are ignored. Existing files are skipped unless overwrite=True.
    """
    input_dir = input_dir.resolve()
    output_dir = output_dir.resolve()
    ensure_dir(output_dir)

    image_paths: list[Path] = [p for p in input_dir.rglob('*') if p.is_file() and is_image_file(p)]

    for src in tqdm(image_paths, desc=f'Processing -> {output_dir.name}', unit='img'):
        rel = src.relative_to(input_dir)
        dst = output_dir / rel
        ensure_dir(dst.parent)
        if dst.exists() and not overwrite:
            continue

        img = cv2.imread(str(src), cv2.IMREAD_COLOR)
        if img is None:
            continue

        out = transform(img)

        # Choose output format by extension; default to PNG if unknown
        ext = dst.suffix.lower()
        if ext not in IMAGE_EXTENSIONS:
            dst = dst.with_suffix('.png')

        cv2.imwrite(str(dst), out)

    print(f'Done. Wrote to: {output_dir}')


In [4]:
# Run both transformations
ensure_dir(OUTPUT_DIR_GLOBAL)
ensure_dir(OUTPUT_DIR_CLAHE)

print('Input :', INPUT_DIR)
print('Global:', OUTPUT_DIR_GLOBAL)
print('CLAHE :', OUTPUT_DIR_CLAHE)

# Global histogram equalization
process_dataset(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR_GLOBAL,
    transform=equalize_global_bgr,
    overwrite=False,
)

# CLAHE
process_dataset(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR_CLAHE,
    transform=lambda img: equalize_clahe_bgr(img, clip_limit=3.0, tile_grid_size=(8, 8)),
    overwrite=False,
)



Input : /Users/Kota/blended/Team3AmazonProject/data/mixed
Global: /Users/Kota/blended/Team3AmazonProject/data/mixed_grobal
CLAHE : /Users/Kota/blended/Team3AmazonProject/data/mixed_clahe


Processing -> mixed_grobal:  94%|█████████▍| 1373/1464 [00:12<00:00, 91.07img/s] libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: cHRM chunk does not match sRGB
Processing -> mixed_grobal: 100%|██████████| 1464/1464 [00:13<00:00, 106.22img/s]


Done. Wrote to: /Users/Kota/blended/Team3AmazonProject/data/mixed_grobal


Processing -> mixed_clahe:  94%|█████████▎| 1369/1464 [00:11<00:00, 105.73img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: cHRM chunk does not match sRGB
Processing -> mixed_clahe: 100%|██████████| 1464/1464 [00:12<00:00, 116.32img/s]

Done. Wrote to: /Users/Kota/blended/Team3AmazonProject/data/mixed_clahe


In [ ]:
# Visualize random samples: Original vs Global vs CLAHE (rows = up to 10 images)
import random
import matplotlib.pyplot as plt

SAMPLE_COUNT = 10

# Collect candidate images from INPUT_DIR (commonly under images/{train,val,test})
all_images = [p for p in INPUT_DIR.rglob('*') if p.is_file() and is_image_file(p)]
random.shuffle(all_images)
samples = all_images[:SAMPLE_COUNT]

n = len(samples)
fig, axes = plt.subplots(nrows=n, ncols=3, figsize=(12, 3*n))
if n == 1:
    axes = np.array([axes])

for row_idx, src in enumerate(samples):
    bgr = cv2.imread(str(src), cv2.IMREAD_COLOR)
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    g_out = equalize_global_bgr(bgr)
    c_out = equalize_clahe_bgr(bgr, clip_limit=3.0, tile_grid_size=(8, 8))

    g_rgb = cv2.cvtColor(g_out, cv2.COLOR_BGR2RGB)
    c_rgb = cv2.cvtColor(c_out, cv2.COLOR_BGR2RGB)

    axes[row_idx, 0].imshow(rgb)
    axes[row_idx, 0].set_title('Original')
    axes[row_idx, 0].axis('off')

    axes[row_idx, 1].imshow(g_rgb)
    axes[row_idx, 1].set_title('Global')
    axes[row_idx, 1].axis('off')

    axes[row_idx, 2].imshow(c_rgb)
    axes[row_idx, 2].set_title('CLAHE')
    axes[row_idx, 2].axis('off')

plt.tight_layout()
plt.show()
